# Technology Production Problem with AMPL in Python
## AMPL Modeling for Book Problem 2.2


## Requirements

This notebook uses `amplpy` together with the HiGHS solver module.

If your environment does not have `amplpy`, install it first:

```python
%pip install amplpy
```

The first call to `ampl_notebook(modules=["highs"], license_uuid="default")` may download the solver module if it is not already available.


In [14]:
from __future__ import annotations

from functools import wraps
from time import perf_counter


In [15]:
%pip install amplpy
try:
    from amplpy import ampl_notebook
except ImportError as exc:
    raise ImportError(
        "This notebook requires amplpy. Install it with `%pip install amplpy` before running."
    ) from exc


def timed(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        result = func(*args, **kwargs)
        elapsed = perf_counter() - start
        return result, elapsed
    return wrapper


Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: /Applications/Xcode.app/Contents/Developer/usr/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [16]:
def create_ampl_instance(solver: str = "highs"):
    ampl = ampl_notebook(modules=[solver], license_uuid="default")
    ampl.option["solver"] = solver
    ampl.option["solver_msg"] = 0
    return ampl


def extract_solution(variable, variable_order):
    values = variable.get_values().to_dict()
    solution = {}

    for name in variable_order:
        if name in values:
            solution[name] = int(round(values[name]))
        else:
            solution[name] = int(round(values[(name,)]))

    return solution


def extract_matrix_solution(variable):
    values = variable.get_values().to_dict()
    solution = {}

    for key, value in values.items():
        if abs(value) > 1e-9:
            solution[key] = int(round(value))

    return solution


## Problem 1: Base Technology Production

Variables:
- `televisions`
- `sound_systems`

Objective:
- maximize weekly profit


In [17]:
BASE_EXPECTED = {
    "televisions": 174,
    "sound_systems": 130,
    "profit": 50400,
}


@timed
def solve_base_technology_production_with_ampl(solver: str = "highs"):
    ampl = create_ampl_instance(solver)
    ampl.eval(
        r'''
        var televisions >= 0 integer;
        var sound_systems >= 0 integer;

        maximize Profit:
            200 * televisions + 120 * sound_systems;

        subject to ProductionAssembly:
            10 * televisions + 12 * sound_systems <= 3300;

        subject to Testing:
            30 * televisions + 6 * sound_systems <= 6000;

        subject to MinimumOrder:
            televisions + sound_systems >= 100;

        subject to AudioLowerBound:
            sound_systems >= 0.25 * televisions;

        subject to AudioUpperBound:
            sound_systems <= televisions + 150;
        '''
    )
    ampl.solve(solver=solver)

    solution = {
        "televisions": int(round(ampl.get_value("televisions"))),
        "sound_systems": int(round(ampl.get_value("sound_systems"))),
        "profit": int(round(ampl.get_value("Profit"))),
    }
    return solution


In [18]:
base_result, base_time = solve_base_technology_production_with_ampl()

print("=== BASE TECHNOLOGY PRODUCTION RESULTS WITH AMPL ===")
print(f"Solution -> {base_result}")
print(f"Time     -> {base_time:.8f} seconds")

assert base_result == BASE_EXPECTED
print("The AMPL solution matches the textbook optimum.")


AMPL Version 20250901 (Darwin-22.6.0, 64-bit)
Demo license with maintenance expiring 20270131.
Using license file "/Users/lfuentesl/Library/Python/3.9/lib/python/site-packages/ampl_module_base/bin/ampl.lic".

HiGHS 1.11.0: === BASE TECHNOLOGY PRODUCTION RESULTS WITH AMPL ===
Solution -> {'televisions': 174, 'sound_systems': 130, 'profit': 50400}
Time     -> 1.79141762 seconds
The AMPL solution matches the textbook optimum.


## Problem 2: Split Production and Assembly Variant

The variant replaces the original shared restriction with two identical half-capacity restrictions.


In [19]:
VARIANT_EXPECTED = {
    "televisions": 174,
    "sound_systems": 130,
    "profit": 50400,
}


@timed
def solve_variant_technology_production_with_ampl(solver: str = "highs"):
    ampl = create_ampl_instance(solver)
    ampl.eval(
        r'''
        var televisions >= 0 integer;
        var sound_systems >= 0 integer;

        maximize Profit:
            200 * televisions + 120 * sound_systems;

        subject to Production:
            5 * televisions + 6 * sound_systems <= 1650;

        subject to Assembly:
            5 * televisions + 6 * sound_systems <= 1650;

        subject to Testing:
            30 * televisions + 6 * sound_systems <= 6000;

        subject to MinimumOrder:
            televisions + sound_systems >= 100;

        subject to AudioLowerBound:
            sound_systems >= 0.25 * televisions;

        subject to AudioUpperBound:
            sound_systems <= televisions + 150;
        '''
    )
    ampl.solve(solver=solver)

    solution = {
        "televisions": int(round(ampl.get_value("televisions"))),
        "sound_systems": int(round(ampl.get_value("sound_systems"))),
        "profit": int(round(ampl.get_value("Profit"))),
    }
    return solution


In [20]:
variant_result, variant_time = solve_variant_technology_production_with_ampl()

print("=== SPLIT PRODUCTION AND ASSEMBLY VARIANT RESULTS WITH AMPL ===")
print(f"Solution -> {variant_result}")
print(f"Time     -> {variant_time:.8f} seconds")

assert variant_result == VARIANT_EXPECTED
print("The AMPL variant confirms that the optimum is unchanged.")


AMPL Version 20250901 (Darwin-22.6.0, 64-bit)
Demo license with maintenance expiring 20270131.
Using license file "/Users/lfuentesl/Library/Python/3.9/lib/python/site-packages/ampl_module_base/bin/ampl.lic".

HiGHS 1.11.0: === SPLIT PRODUCTION AND ASSEMBLY VARIANT RESULTS WITH AMPL ===
Solution -> {'televisions': 174, 'sound_systems': 130, 'profit': 50400}
Time     -> 1.53143950 seconds
The AMPL variant confirms that the optimum is unchanged.


In [21]:
print("=== FINAL COMPARISON ===")
print()
print("BASE PROBLEM")
print(f"Solution -> {base_result}")
print(f"Time     -> {base_time:.8f} seconds")
print()
print("VARIANT PROBLEM")
print(f"Solution -> {variant_result}")
print(f"Time     -> {variant_time:.8f} seconds")
print()


=== FINAL COMPARISON ===

BASE PROBLEM
Solution -> {'televisions': 174, 'sound_systems': 130, 'profit': 50400}
Time     -> 1.79141762 seconds

VARIANT PROBLEM
Solution -> {'televisions': 174, 'sound_systems': 130, 'profit': 50400}
Time     -> 1.53143950 seconds

